# 🧪 02 · AI Test-Case Generator

**Level:** Everyone · **Time:** ~20 min

Give the AI a **requirement / user story** → get back structured **test cases**:
positive, negative, edge, and boundary — as a clean table you can export.

Free, in the cloud, no API key.  ▶️ Run each cell with **Shift+Enter**.

> ⚙️ Enable GPU: *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Setup (install + load model)

In [ ]:
!pip install -q -U transformers accelerate
import torch, json, re
from transformers import pipeline

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'  # tip: try 'Qwen/Qwen2.5-3B-Instruct' for better quality
chat = pipeline('text-generation', model=MODEL,
                torch_dtype=torch.bfloat16, device_map='auto')
print('✅ ready')

### Step 2 — The QA prompt
This is the heart of the tool. We give the model a **clear role**, **exact instructions**,
and ask it to reply in **JSON** so we can turn it into a table. This is the part your
community should study and tweak — better prompt = better test cases.

In [ ]:
SYSTEM = '''You are a senior QA engineer. You write thorough, practical test cases.
For a given requirement, produce test cases covering POSITIVE, NEGATIVE, EDGE, and BOUNDARY scenarios.
Return ONLY valid JSON (no prose, no markdown fences) shaped exactly as:
{"test_cases": [
  {"id": "TC-01", "title": "...", "type": "positive|negative|edge|boundary",
   "priority": "high|medium|low", "steps": ["..."], "expected": "..."}
]}'''

def generate_test_cases(requirement, max_new_tokens=1200):
    user = f'Requirement:\n{requirement}\n\nGenerate 6-10 test cases as JSON.'
    out = chat([{'role':'system','content':SYSTEM}, {'role':'user','content':user}],
               max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]['generated_text'][-1]['content']
    # be forgiving: pull the JSON object out even if the model adds stray text
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group(0)) if m else {'test_cases': [], 'raw': text}

print('✅ generator ready')

### Step 3 — Generate! ✨
Edit `requirement` to anything from your own backlog.

In [ ]:
requirement = 'As a user, I can reset my password by requesting an email link that expires in 15 minutes.'

result = generate_test_cases(requirement)
print(f"Generated {len(result['test_cases'])} test cases")

### Step 4 — See them as a table

In [ ]:
import pandas as pd
df = pd.DataFrame(result['test_cases'])
if 'steps' in df: df['steps'] = df['steps'].apply(lambda s: ' → '.join(s) if isinstance(s, list) else s)
df

### Step 5 — Export (Markdown / CSV)
Take the output straight into Jira, TestRail, a wiki, or a spreadsheet.

In [ ]:
df.to_csv('test_cases.csv', index=False)
with open('test_cases.md','w') as f: f.write(df.to_markdown(index=False))
print('✅ saved test_cases.csv and test_cases.md  (see the Files panel on the left)')
print(df.to_markdown(index=False))

---
### 🧠 Discussion for the group
- Did it find edge cases *you* would have missed? Did it invent any wrong ones?
- **This is the key lesson:** AI is a *fast first draft*, not a replacement for a tester's judgement.
- Try improving `SYSTEM` (e.g. ask for Gherkin, or for test data values). Re-run and compare.

👉 Next: **`03_custom_qa_assistant.ipynb`** — shape the model into a dedicated QA-Bot.